# Fabric Notebook AI Auditor v1

_Thin orchestrator over `fabric-scanner==0.3.9`._

Runs Fabric AI Functions over every notebook in a lakehouse
export folder and scores each for **external-resource access**
and **data-exfiltration risk**.

**⚠ Privacy boundary.** Notebook content is sent to Fabric AI
Functions for evaluation. Only run on notebook corpora that are
already permitted to flow to the Fabric AI model endpoint
configured for your workspace.

Five cells:
1. **Install** the scanner wheel.
2. **Configure** the scan + AI options (the only cells most users touch).
3. **Probe** the source — sanity-check paths.
4. **Run** the AI audit — writes the chunks + results Delta tables.
5. **Explore** the results — top risk + cross-table JOIN with
   the rule-based findings table.

All scanner + AI logic lives in the installable wheel. Edits to
the AI prompt, response schema, or aggregation happen there —
not in this notebook. `PROMPT_VERSION` is stamped on every
output row so you can correlate scores back to the prompt that
produced them.

In [ ]:
# Install the scanner wheel (re-run when you bump the version).
%pip install -q fabric-scanner==0.3.9

## 1. Configuration

`ScannerConfig` controls source/target resolution (same as the
rule-based scanner); `AIAuditOptions` controls chunk size,
per-notebook caps, and the optional hard budget.

In [ ]:
from fabric_scanner import ScannerConfig
from fabric_scanner.ai import AIAuditOptions, PROMPT_VERSION

cfg = ScannerConfig(
    # --- Where to read notebooks from (lakehouse-only in v1) ---
    source_mode      = "lakehouse",
    source_layout    = "ws_dated",   # or "flat"
    source_subpath   = "Files/notebook_exports",
    source_file_glob = "*.ipynb",
    # Leave source_lakehouse_* blank to auto-detect the mounted
    # default lakehouse (recommended).
    # source_lakehouse_workspace_id = "<ws-guid>",
    # source_lakehouse_id           = "<lh-guid>",

    # --- Where to write outputs ---
    write_to_default_lakehouse = True,
    inventory_table = "v2_inventory",
    output_table    = "v2_findings",  # unused by the AI auditor

    # --- Behavior ---
    max_notebooks         = 0,    # 0 = no cap (start small for cost!)
    target_partition_size = 200,
)

opts = AIAuditOptions(
    ai_output_table = "notebook_ai_audit_results",
    ai_chunks_table = "notebook_ai_audit_chunks",
    # Per-notebook chunk size + caps. Lower these if you're
    # rate-limited or want a cheaper smoke run.
    ai_max_content_chars       = 14_000,
    ai_max_chunks_per_notebook = 50,
    # Hard budget across the whole run. None = unlimited (risky).
    # ai_max_total_chunks      = 500,
    write_chunks_table = True,
)
print(f"Prompt version : {PROMPT_VERSION}")
print(f"Run id         : {opts.run_id}")

## 2. Diagnostics — probe paths

Prints a one-screen summary of what the AI auditor *will* read.
Safe to run before kicking off the actual AI calls.

In [ ]:
from fabric_scanner import resolve_paths, probe

resolved = resolve_paths(cfg)
probe(cfg, resolved)

## 3. Run the AI audit

Builds the inventory, chunks each notebook, calls
`df.ai.generate_response` on every chunk, defensively parses
the JSON, and aggregates to one row per notebook.

`run_ai_audit` raises `BudgetExceededError` if the planned chunk
count exceeds `opts.ai_max_total_chunks` (when set). It also
raises a friendly error if Fabric AI Functions aren't available
in the current runtime.

In [ ]:
from fabric_scanner.ai import run_ai_audit

result = run_ai_audit(cfg, opts, spark)

print(f"Notebooks audited : {result.notebooks_count:,}")
print(f"Chunks total      : {result.chunks_total:,}")
print(f"Chunks parsed     : {result.chunks_parsed:,}")
print(f"Chunks errored    : {result.chunks_errored:,}")
print(f"Results table     : {result.results_table}")
print(f"Chunks table      : {result.chunks_table}")
print(f"Run id            : {result.run_id}")

## 4. Explore the results

`result.results_df` is one row per notebook with max scores
across all chunks. `result.chunks_df` is the underlying
per-chunk detail — useful when investigating a specific
high-risk score.

In [ ]:
# Highest-risk notebooks first.
from pyspark.sql import functions as F
display(result.results_df.orderBy(
    F.col("exfiltration_risk_score").desc_nulls_last(),
    F.col("external_resource_access_score").desc_nulls_last(),
))

In [ ]:
# Drill into the chunks for the top notebook.
top_id = (result.results_df
          .orderBy(F.col("exfiltration_risk_score").desc_nulls_last())
          .limit(1).select("notebook_id").collect()[0]["notebook_id"])
display(result.chunks_df
        .filter(F.col("notebook_id") == top_id)
        .orderBy("chunk_index"))

## 5. JOIN with the rule-based findings

Both tables share `(workspace_id, source_dated_partition,
display_name)` as a stable join key, plus `content_hash` for
extra precision when display_names collide across exports.

In [ ]:
ai = spark.table(opts.ai_output_table)
findings = spark.table(cfg.output_table)

joined = (findings.alias("r")
    .join(ai.alias("a"),
        (F.col("r.workspace_id")           == F.col("a.workspace_id")) &
        (F.col("r.source_dated_partition") == F.col("a.source_dated_partition")) &
        (F.col("r.display_name")           == F.col("a.display_name")),
        how="left")
    .select(
        "r.workspace_id", "r.display_name", "r.severity",
        "r.finding_type", "r.pattern_label",
        F.col("a.exfiltration_risk_score").alias("exfil"),
        F.col("a.external_resource_access_score").alias("ext_access"),
        F.col("a.top_rationale").alias("rationale"),
    ))
display(joined.orderBy(F.col("exfil").desc_nulls_last()))